# Fetch named / interesting stars

Prototype notebook. Raw IDs and source columns stay in table so parsing/merge choices can be changed without re-querying everything.


In [80]:
import json
import re

import numpy as np
import pandas as pd
from pathlib import Path
from astropy.table import vstack, QTable, Table
from astroquery.gaia import Gaia
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier

pd.set_option("display.max_columns", None)



In [2]:
HYG_URL = "https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv"
SIMBAD_ROW_LIMIT = 30000
GAIA_BATCH_SIZE = 1000

# Unnamed stars to lookup
INTERESTING_SIMBAD_LOOKUPS = [
    "WR 134",
    "WR 135",
    "WR 102",
    "2MASS J07464256+2000321",
    "Gliese 229B",
    "WISE J085510.83-071442.5", # WISE 0855-0714
    # "Y Canum Venaticorum", # already in as La Superba
    "R Leporis",
    "HD 26",
    "R Coronae Borealis",
    "GD 358",
    "KPD 0005+5106",
    "GJ 1117",
    "WD 0343+247",
    "G29-38",
    "Grw +70 8247",
    "ZZ Ceti",
    "P Cygni",
]


## Multi-star handling

Default policy: one row per physical child star when catalog has a child/component entry with its own IDs/astrometry/spectral type. Keep a system row only when no child rows are available. This avoids collapsing objects like Luhman 16 A/B. Some systems still need case-by-case cleanup if SIMBAD/HYG only expose a blended/system entry.


## Load HYG named / Bayer stars first

HYG `ra` is hours, `dec` degrees, `dist` parsecs. Convert RA to degrees and parallax to mas.


In [3]:
hyg = pd.read_csv(HYG_URL)

hyg_named = hyg[hyg["proper"].notna() | hyg["bayer"].notna()].copy()

hyg_seed = pd.DataFrame()
hyg_seed["source_catalog"] = "HYG"
hyg_seed["common_name"] = hyg_named["proper"]
hyg_seed["bayer_name"] = None
hyg_seed["simbad_bayer"] = np.where(hyg_named["bayer"].notna(), hyg_named["bayer"].astype(str) + " " + hyg_named["con"].astype(str), None)
hyg_seed["HD_id"] = hyg_named["hd"].dropna().astype("Int64").astype(str).reindex(hyg_named.index)
hyg_seed["HIP_id"] = hyg_named["hip"].dropna().astype("Int64").astype(str).reindex(hyg_named.index)
hyg_seed["gliese_id"] = hyg_named["gl"].where(hyg_named["gl"].notna(), None)
hyg_seed["gaia_dr3_id"] = None
hyg_seed["gaia_dr2_id"] = None
hyg_seed["2mass_id"] = None
hyg_seed["WR_id"] = None
hyg_seed["sp_type_hyg"] = hyg_named["spect"]
hyg_seed["ra_hyg"] = hyg_named["ra"] * 15.0
hyg_seed["dec_hyg"] = hyg_named["dec"]
hyg_seed["parallax_hyg"] = 1000.0 / hyg_named["dist"]
hyg_seed["radius_hyg"] = np.nan
hyg_seed["luminosity_hyg"] = hyg_named["lum"]
hyg_seed["BV_color_index_hyg"] = hyg_named["ci"]
hyg_seed["dist_hyg"] = hyg_named["dist"]
hyg_seed["hyg_id"] = hyg_named["id"].astype(str)

print(f"HYG named/Bayer rows: {len(hyg_seed)}")
hyg_seed


HYG named/Bayer rows: 1687


,source_catalog,common_name,bayer_name,simbad_bayer,HD_id,HIP_id,gliese_id,gaia_dr3_id,gaia_dr2_id,2mass_id,WR_id,sp_type_hyg,ra_hyg,dec_hyg,parallax_hyg,radius_hyg,luminosity_hyg,BV_color_index_hyg,dist_hyg,hyg_id
0,NaN,Sol,None,None,NaN,NaN,None,None,None,None,None,G2V,0.000000,0.000000,inf,NaN,1.000000e+00,0.656,0.0000,0
88,NaN,NaN,None,Tau Phe,224834,88,None,None,None,None,None,G8III,0.269115,-48.809876,5.499999,NaN,1.496925e+02,0.911,181.8182,88
122,NaN,NaN,None,The Oct,224889,122,None,None,None,None,None,K2III,0.399240,-77.065724,15.019999,NaN,4.729334e+01,1.254,66.5779,122
183,NaN,NaN,None,Zet Scl,224990,183,None,None,None,None,None,B4V,0.583005,-29.720414,6.490000,NaN,1.993425e+02,-0.150,154.0832,183
676,NaN,Alpheratz,None,Alp And,358,677,None,None,None,None,None,B9p,2.096865,29.090432,33.620000,NaN,1.144986e+02,-0.038,29.7442,676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119254,NaN,NaN,None,Gam CrA,177475,NaN,Gl 743.1B,None,None,None,None,F8 V,286.602705,-37.064740,55.889964,NaN,2.787405e+00,NaN,17.8923,119259
119428,NaN,NaN,None,Mu-2 Cyg,206827,NaN,Gl 836.6B,None,None,None,None,G2 V,326.032635,28.743078,44.640067,NaN,1.601032e+00,NaN,22.4014,119433
119513,NaN,EZ Aqr,None,None,NaN,NaN,Gl 866A,None,None,None,None,M5 e,339.638430,-15.302542,294.299420,NaN,8.677607e-05,1.980,3.3979,119518
119580,NaN,Ross 248,None,None,NaN,NaN,Gl 905,None,None,None,None,dM6 e,355.475685,44.174924,315.596794,NaN,1.060718e-04,1.900,3.1686,119585


## Query SIMBAD named/Bayer stars

One row per SIMBAD object. Identifier joins are aggregated into ID columns.


In [4]:
simbad_query = f"""
SELECT TOP {SIMBAD_ROW_LIMIT}
  b.oid,
  b.main_id,
  b.ra,
  b.dec,
  b.plx_value,
  b.plx_err,
  b.sp_type,
  b.otype,
  max(nameid.id) AS name_id,
  max(gaia3.id) AS gaia_dr3_id,
  max(gaia2.id) AS gaia_dr2_id,
  max(hd.id) AS hd_id,
  max(hip.id) AS hip_id,
  max(gliese.id) AS gliese_id,
  max(tmass.id) AS tmass_id,
  max(wr.id) AS wr_id,
  max(bayer.id) AS simbad_bayer
FROM basic AS b
LEFT JOIN ident AS nameid
  ON b.oid = nameid.oidref
  AND nameid.id LIKE 'NAME %'
LEFT JOIN ident AS gaia3
  ON b.oid = gaia3.oidref
  AND gaia3.id LIKE 'Gaia DR3 %'
LEFT JOIN ident AS gaia2
  ON b.oid = gaia2.oidref
  AND gaia2.id LIKE 'Gaia DR2 %'
LEFT JOIN ident AS hd
  ON b.oid = hd.oidref
  AND hd.id LIKE 'HD %'
LEFT JOIN ident AS hip
  ON b.oid = hip.oidref
  AND hip.id LIKE 'HIP %'
LEFT JOIN ident AS gliese
  ON b.oid = gliese.oidref
  AND (gliese.id LIKE 'GJ %' OR gliese.id LIKE 'Gl %' OR gliese.id LIKE 'GL %' OR gliese.id LIKE 'Gliese %')
LEFT JOIN ident AS tmass
  ON b.oid = tmass.oidref
  AND tmass.id LIKE '2MASS %'
LEFT JOIN ident AS wr
  ON b.oid = wr.oidref
  AND wr.id LIKE 'WR %'
LEFT JOIN ident AS bayer
  ON b.oid = bayer.oidref
  AND (bayer.id LIKE '* alf%' OR
       bayer.id LIKE '* bet%' OR
       bayer.id LIKE '* gam%' OR
       bayer.id LIKE '* del%' OR
       bayer.id LIKE '* eps%' OR
       bayer.id LIKE '* zet%' OR
       bayer.id LIKE '* eta%' OR
       bayer.id LIKE '* the%' OR
       bayer.id LIKE '* iot%' OR
       bayer.id LIKE '* kap%' OR
       bayer.id LIKE '* lam%' OR
       bayer.id LIKE '* mu.%' OR
       bayer.id LIKE '* nu.%' OR
       bayer.id LIKE '* ksi%' OR
       bayer.id LIKE '* omi%' OR
       bayer.id LIKE '* pi.%' OR
       bayer.id LIKE '* rho%' OR
       bayer.id LIKE '* sig%' OR
       bayer.id LIKE '* tau%' OR
       bayer.id LIKE '* ups%' OR
       bayer.id LIKE '* phi%' OR
       bayer.id LIKE '* khi%' OR
       bayer.id LIKE '* psi%' OR
       bayer.id LIKE '* ome%')
WHERE (nameid.id IS NOT NULL OR bayer.id IS NOT NULL)
  AND b.sp_type IS NOT NULL
GROUP BY b.oid, b.main_id, b.ra, b.dec, b.plx_value, b.plx_err, b.sp_type, b.otype
ORDER BY main_id
"""

simbad_raw = Simbad.query_tap(simbad_query).to_pandas()
print(f"SIMBAD named/Bayer rows: {len(simbad_raw)}")
simbad_raw


SIMBAD named/Bayer rows: 2606


,oid,main_id,ra,dec,plx_value,plx_err,sp_type,otype,name_id,gaia_dr3_id,gaia_dr2_id,hd_id,hip_id,gliese_id,tmass_id,wr_id,simbad_bayer
0,390148,* 11 UMi,229.274539,71.823902,7.9260,0.0874,K4III,*,NAME Pherkad Minor,Gaia DR3 1696798367260229376,Gaia DR2 1696798367260229376,HD 136726,HIP 74793,,2MASS J15170588+7149258,,
1,1554417,* 14 And,352.822555,39.236199,13.1681,0.0727,G8III,PM*,NAME Veritate,Gaia DR3 1920113512486282240,Gaia DR2 1920113512486282240,HD 221345,HIP 116076,,2MASS J23311742+3914102,,
2,661336,* 16 Tau,56.200896,24.289468,7.3852,0.0724,B7V,*,NAME Celeno,Gaia DR3 65287458566524928,Gaia DR2 65287458566524928,HD 23288,HIP 17489,,2MASS J03444821+2417222,,
3,1930345,* 17 Com,187.227926,25.912852,12.9991,0.3721,B9VCrEu,a2*,NAME ** STFA 21,Gaia DR3 3960724252307532672,Gaia DR2 3960724252307181056,HD 108662,HIP 60904,,2MASS J12285470+2554463,,
4,661210,* 17 Tau,56.218905,24.113338,8.3457,0.4386,B6IIIe,Be*,NAME Electra,Gaia DR3 65271996684817280,Gaia DR2 65271996684277504,HD 23302,HIP 17499,,2MASS J03445253+2406478,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2601,3280716,* zet Tuc,5.017744,-64.874794,116.1826,0.1334,F9.5V,PM*,,Gaia DR3 4900108950849461248,Gaia DR2 4900108950847893376,HD 1581,HIP 1599,GJ 17,2MASS J00200446-6452282,,* zet Tuc
2602,365180,* zet UMi,236.014661,77.794493,9.0827,0.1422,A3Vn,V*,,Gaia DR3 1707399209837352064,Gaia DR2 1707399209836866560,HD 142105,HIP 77055,,,,* zet UMi
2603,2168382,* zet Vir,203.673297,-0.595816,43.7467,0.3330,A2Van,PM*,NAME Heze,Gaia DR3 3662636823132300032,Gaia DR2 3662636823132020480,HD 118098,HIP 66249,GJ 3792,2MASS J13344160-0035450,,* zet Vir
2604,5331611,* zet Vir B,203.673750,-0.596444,NaN,NaN,M4V-M7V,LM*,,,,,,,,,* zet Vir B


## Lookup interesting unnamed stars by SIMBAD name

Queried name becomes `common_name` for that row. Edit `INTERESTING_SIMBAD_LOOKUPS` above.


In [5]:
interesting_names_sql = ", ".join("'" + name.replace("'", "''") + "'" for name in INTERESTING_SIMBAD_LOOKUPS)

interesting_query = f"""
SELECT
  q.name AS lookup_name,
  b.oid,
  b.main_id,
  b.ra,
  b.dec,
  b.plx_value,
  b.plx_err,
  b.sp_type,
  b.otype,
  max(gaia3.id) AS gaia_dr3_id,
  max(gaia2.id) AS gaia_dr2_id,
  max(hd.id) AS hd_id,
  max(hip.id) AS hip_id,
  max(gliese.id) AS gliese_id,
  max(tmass.id) AS tmass_id,
  max(wr.id) AS wr_id,
  max(bayer.id) AS simbad_bayer
FROM TAP_UPLOAD.lookup AS q
JOIN ident AS requested ON requested.id = q.name
JOIN basic AS b ON b.oid = requested.oidref
LEFT JOIN ident AS gaia3 ON b.oid = gaia3.oidref AND gaia3.id LIKE 'Gaia DR3 %'
LEFT JOIN ident AS gaia2 ON b.oid = gaia2.oidref AND gaia2.id LIKE 'Gaia DR2 %'
LEFT JOIN ident AS hd ON b.oid = hd.oidref AND hd.id LIKE 'HD %'
LEFT JOIN ident AS hip ON b.oid = hip.oidref AND hip.id LIKE 'HIP %'
LEFT JOIN ident AS gliese ON b.oid = gliese.oidref AND (gliese.id LIKE 'GJ %' OR gliese.id LIKE 'Gl %' OR gliese.id LIKE 'GL %' OR gliese.id LIKE 'Gliese %')
LEFT JOIN ident AS tmass ON b.oid = tmass.oidref AND tmass.id LIKE '2MASS %'
LEFT JOIN ident AS wr ON b.oid = wr.oidref AND wr.id LIKE 'WR %'
LEFT JOIN ident AS bayer ON b.oid = bayer.oidref AND bayer.id LIKE '* ___%'
GROUP BY q.name, b.oid, b.main_id, b.ra, b.dec, b.plx_value, b.plx_err, b.sp_type, b.otype
"""

lookup_upload = Table({"name": INTERESTING_SIMBAD_LOOKUPS})
interesting_raw = Simbad.query_tap(interesting_query, lookup=lookup_upload).to_pandas()
print(f"Interesting lookup rows: {len(interesting_raw)}")
interesting_raw



Interesting lookup rows: 17


,lookup_name,oid,main_id,ra,dec,plx_value,plx_err,sp_type,otype,gaia_dr3_id,gaia_dr2_id,hd_id,hip_id,gliese_id,tmass_id,wr_id,simbad_bayer
0,2MASS J07464256+2000321,1005244,LSPM J0746+2000,116.677342,20.008940,80.9000,0.8000,L0+L1.5,BD*,Gaia DR3 672055699133504768,Gaia DR2 672055699133504768,,,,2MASS J07464256+2000321,,
1,G29-38,1402376,V* ZZ Psc,352.198485,5.248399,57.0620,0.0251,DA4.1,WD*,Gaia DR3 2660358032257156736,Gaia DR2 2660358032257156736,,,GJ 895.2,2MASS J23284760+0514540,,
2,GD 358,4015980,GD 358,251.826635,32.475795,23.2441,0.0240,DB2,WD*,Gaia DR3 1314045729544380288,Gaia DR2 1314045729544380288,,,,2MASS J16471839+3228328,,
3,GJ 1117,1122002,Ton 368,134.811308,32.953377,43.3637,0.0326,DQ6,WD*,Gaia DR3 712888090655562624,Gaia DR2 712888090655562624,,,GJ 1117,2MASS J08591476+3257121,,
4,Gliese 229B,864867,HD 42581B,92.645000,-21.866667,173.1900,1.1200,T6.5,BD*,,,HD 42581B,,GJ 229 B,,,
5,Grw +70 8247,300191,LAWD 73,285.042722,70.664282,77.6696,0.0161,DAP4.5,WD*,Gaia DR3 2262849634963004416,Gaia DR2 2262849634963004416,,,GJ 742,2MASS J19001024+7039512,,
6,HD 26,1423061,HD 26,1.342530,8.787804,3.2044,0.0309,C-H1.5,SB*,Gaia DR3 2752856200291132416,Gaia DR2 2752856200291132416,HD 26,HIP 447,,2MASS J00052221+0847158,,
7,KPD 0005+5106,44218,KPD 0005+5106,2.075710,51.387944,2.4089,0.0355,DOZ1,WD*,Gaia DR3 395429431270137728,Gaia DR2 395429431270137728,,,,2MASS J00081816+5123165,,
8,P Cygni,2954633,* P Cyg,304.446675,38.032930,0.6251,0.0729,B1-2Ia-0ep,s*b,Gaia DR3 2061242908036996352,Gaia DR2 2061242908036996352,HD 193237,HIP 100044,,2MASS J20174719+3801585,,* P Cyg
9,R Coronae Borealis,2831935,V* R CrB,237.143398,28.156749,1.4366,0.2823,G0Iep,RC*,Gaia DR3 1224565824009709312,Gaia DR2 1224565824009017600,HD 141527,HIP 77442,,2MASS J15483440+2809242,,


## Parse SIMBAD/HYG IDs and names


In [6]:
GREEK_ABBR = {
    "alf": "α", "bet": "β", "gam": "γ", "del": "δ", "eps": "ε", "zet": "ζ", "eta": "η", "the": "θ",
    "iot": "ι", "kap": "κ", "lam": "λ", "mu.": "μ", "nu.": "ν", "ksi": "ξ", "omi": "ο", "pi.": "π",
    "rho": "ρ", "sig": "σ", "tau": "τ", "ups": "υ", "phi": "φ", "khi": "χ", "psi": "ψ", "ome": "ω",
}
SUPERSCRIPT_DIGITS = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")
CONSTELLATION_GENITIVE = {
    "And": "Andromedae", "Ant": "Antliae", "Aps": "Apodis", "Aql": "Aquilae", "Aqr": "Aquarii", "Ara": "Arae",
    "Ari": "Arietis", "Aur": "Aurigae", "Boo": "Bootis", "CMa": "Canis Majoris", "CMi": "Canis Minoris",
    "CVn": "Canum Venaticorum", "Cae": "Caeli", "Cam": "Camelopardalis", "Cap": "Capricorni", "Car": "Carinae",
    "Cas": "Cassiopeiae", "Cen": "Centauri", "Cep": "Cephei", "Cet": "Ceti", "Cha": "Chamaeleontis", "Cir": "Circini",
    "Cnc": "Cancri", "Col": "Columbae", "Com": "Comae Berenices", "CrA": "Coronae Australis", "CrB": "Coronae Borealis",
    "Crt": "Crateris", "Cru": "Crucis", "Crv": "Corvi", "Cyg": "Cygni", "Del": "Delphini", "Dor": "Doradus",
    "Dra": "Draconis", "Equ": "Equulei", "Eri": "Eridani", "For": "Fornacis", "Gem": "Geminorum", "Gru": "Gruis",
    "Her": "Herculis", "Hor": "Horologii", "Hya": "Hydrae", "Hyi": "Hydri", "Ind": "Indi", "LMi": "Leonis Minoris",
    "Lac": "Lacertae", "Leo": "Leonis", "Lep": "Leporis", "Lib": "Librae", "Lup": "Lupi", "Lyn": "Lyncis",
    "Lyr": "Lyrae", "Men": "Mensae", "Mic": "Microscopii", "Mon": "Monocerotis", "Mus": "Muscae", "Nor": "Normae",
    "Oct": "Octantis", "Oph": "Ophiuchi", "Ori": "Orionis", "Pav": "Pavonis", "Peg": "Pegasi", "Per": "Persei",
    "Phe": "Phoenicis", "Pic": "Pictoris", "PsA": "Piscis Austrini", "Psc": "Piscium", "Pup": "Puppis", "Pyx": "Pyxidis",
    "Ret": "Reticuli", "Scl": "Sculptoris", "Sco": "Scorpii", "Sct": "Scuti", "Ser": "Serpentis", "Sex": "Sextantis",
    "Sge": "Sagittae", "Sgr": "Sagittarii", "Tau": "Tauri", "Tel": "Telescopii", "TrA": "Trianguli Australis",
    "Tri": "Trianguli", "Tuc": "Tucanae", "UMa": "Ursae Majoris", "UMi": "Ursae Minoris", "Vel": "Velorum",
    "Vir": "Virginis", "Vol": "Volantis", "Vul": "Vulpeculae",
}

simbad_seed = simbad_raw.copy()
simbad_seed["source_catalog"] = "SIMBAD"
simbad_seed["common_name"] = simbad_seed["name_id"].str.replace(r"^NAME\s+", "", regex=True)
simbad_seed["gaia_dr3_id"] = simbad_seed["gaia_dr3_id"].str.extract(r"Gaia DR3 (\d+)")[0]
simbad_seed["gaia_dr2_id"] = simbad_seed["gaia_dr2_id"].str.extract(r"Gaia DR2 (\d+)")[0]
simbad_seed["HD_id"] = simbad_seed["hd_id"].str.extract(r"HD\s+(\d+)")[0]
simbad_seed["HIP_id"] = simbad_seed["hip_id"].str.extract(r"HIP\s+(\d+)")[0]
simbad_seed["2mass_id"] = simbad_seed["tmass_id"].str.replace(r"^2MASS\s+", "", regex=True)
simbad_seed["WR_id"] = simbad_seed["wr_id"]
simbad_seed["sp_type_simbad"] = simbad_seed["sp_type"]
simbad_seed["ra_simbad"] = simbad_seed["ra"]
simbad_seed["dec_simbad"] = simbad_seed["dec"]
simbad_seed["parallax_simbad"] = simbad_seed["plx_value"]
simbad_seed["radius_simbad"] = np.nan
simbad_seed["luminosity_simbad"] = np.nan
simbad_seed["BV_color_index_simbad"] = np.nan
simbad_seed["dist_simbad"] = 1000.0 / simbad_seed["plx_value"]
simbad_seed["simbad_bayer"] = simbad_seed["simbad_bayer"].str.replace(r"^\*\s+", "", regex=True)

interesting_seed = interesting_raw.copy()
interesting_seed["source_catalog"] = "SIMBAD_LOOKUP"
interesting_seed["common_name"] = interesting_seed["lookup_name"]
interesting_seed["gaia_dr3_id"] = interesting_seed["gaia_dr3_id"].str.extract(r"Gaia DR3 (\d+)")[0]
interesting_seed["gaia_dr2_id"] = interesting_seed["gaia_dr2_id"].str.extract(r"Gaia DR2 (\d+)")[0]
interesting_seed["HD_id"] = interesting_seed["hd_id"].str.extract(r"HD\s+(\d+)")[0]
interesting_seed["HIP_id"] = interesting_seed["hip_id"].str.extract(r"HIP\s+(\d+)")[0]
interesting_seed["2mass_id"] = interesting_seed["tmass_id"].str.replace(r"^2MASS\s+", "", regex=True)
interesting_seed["WR_id"] = interesting_seed["wr_id"]
interesting_seed["sp_type_simbad"] = interesting_seed["sp_type"]
interesting_seed["ra_simbad"] = interesting_seed["ra"]
interesting_seed["dec_simbad"] = interesting_seed["dec"]
interesting_seed["parallax_simbad"] = interesting_seed["plx_value"]
interesting_seed["radius_simbad"] = np.nan
interesting_seed["luminosity_simbad"] = np.nan
interesting_seed["BV_color_index_simbad"] = np.nan
interesting_seed["dist_simbad"] = 1000.0 / interesting_seed["plx_value"]
interesting_seed["simbad_bayer"] = interesting_seed["simbad_bayer"].str.replace(r"^\*\s+", "", regex=True)

all_seed = pd.concat([hyg_seed, simbad_seed, interesting_seed], ignore_index=True, sort=False)

bayer_names = []
for simbad_bayer in all_seed["simbad_bayer"]:
    bayer_name = None
    if isinstance(simbad_bayer, str):
        match = re.match(r"^([a-z.]{3})(\d{0,2})\s+([A-Z][A-Za-z]{2})(?:\s+([A-Z]))?$", simbad_bayer)
        if match:
            greek_abbr, number, constellation_abbr, component = match.groups()
            greek = GREEK_ABBR.get(greek_abbr)
            constellation = CONSTELLATION_GENITIVE.get(constellation_abbr)
            if greek and constellation:
                suffix = "" if component else (str(int(number)).translate(SUPERSCRIPT_DIGITS) if number else "")
                component_suffix = f" {component}" if component else ""
                bayer_name = f"{greek}{suffix} {constellation}{component_suffix}"
    bayer_names.append(bayer_name)
all_seed["bayer_name"] = all_seed["bayer_name"].combine_first(pd.Series(bayer_names, index=all_seed.index))

all_seed["component"] = all_seed["simbad_bayer"].astype(str).str.extract(r"\s([A-Z])$")[0]

# Dedupe by connected identifiers, not one preferred key.
# Example: HYG row may only have HD 212496, while SIMBAD row has Gaia+HD 212496.
# Both must collapse to one object via shared HD alias.
alias_to_key = {}
row_keys = []

for idx, row in all_seed.iterrows():
    aliases = []
    for col, prefix in [
        ("gaia_dr3_id", "Gaia DR3"),
        ("gaia_dr2_id", "Gaia DR2"),
        ("HIP_id", "HIP"),
        ("HD_id", "HD"),
        ("gliese_id", "Gliese"),
        ("2mass_id", "2MASS"),
        ("WR_id", "WR"),
        ("oid", "SIMBAD OID"),
    ]:
        value = row.get(col)
        if pd.notna(value) and str(value).strip() and str(value).lower() != "nan":
            aliases.append(f"{prefix}:{str(value).strip()}")

    if not aliases:
        common = row.get("common_name")
        component = row.get("component")
        aliases.append(f"NAME:{str(common).lower()}:{component if pd.notna(component) else ''}")

    existing_keys = [alias_to_key[alias] for alias in aliases if alias in alias_to_key]
    object_key = sorted(existing_keys)[0] if existing_keys else aliases[0]

    # Merge any previously separate keys now connected by this row.
    for old_key in set(existing_keys):
        if old_key != object_key:
            for alias, key in list(alias_to_key.items()):
                if key == old_key:
                    alias_to_key[alias] = object_key

    for alias in aliases:
        alias_to_key[alias] = object_key
    row_keys.append(object_key)

all_seed["dedupe_key"] = row_keys
all_seed["dedupe_key"] = all_seed["dedupe_key"].map(lambda key: alias_to_key.get(key, key))

all_seed["source_rank"] = all_seed["source_catalog"].map({"SIMBAD_LOOKUP": 0, "SIMBAD": 1, "HYG": 2}).fillna(9)
seed = all_seed.sort_values("source_rank").groupby("dedupe_key", as_index=False).first()

# Keep best human common name separately. Prefer non-ID-looking names; preserve HYG proper names when SIMBAD has only catalog-ish names.
weird_seed_name = all_seed["common_name"].fillna("").str.match(
    r"^(Gaia|HD|HIP|2MASS|WISE|TYC|TIC|UCAC|USNO|GJ|Gl|LP|LHS|LTT|BD|CD|CPD|SAO|IRAS|ASAS|SDSS|2MUCD)",
    case=False,
)
all_seed["common_name_rank"] = np.select(
    [all_seed["common_name"].notna() & ~weird_seed_name & all_seed["source_catalog"].eq("SIMBAD_LOOKUP"),
     all_seed["common_name"].notna() & ~weird_seed_name & all_seed["source_catalog"].eq("HYG"),
     all_seed["common_name"].notna() & ~weird_seed_name,
     all_seed["common_name"].notna()],
    [0, 1, 2, 3],
    default=9,
)
best_common_name = all_seed.sort_values("common_name_rank").groupby("dedupe_key")["common_name"].first()
seed["common_name"] = seed["dedupe_key"].map(best_common_name).combine_first(seed["common_name"])

print(f"Combined seed rows before/after connected-ID dedupe: {len(all_seed)} -> {len(seed)}")
seed[["source_catalog", "common_name", "bayer_name", "simbad_bayer", "gaia_dr3_id", "gaia_dr2_id", "HD_id", "HIP_id", "gliese_id", "2mass_id", "WR_id", "sp_type_hyg", "sp_type_simbad"]]



Combined seed rows before/after connected-ID dedupe: 4310 -> 2736


,source_catalog,common_name,bayer_name,simbad_bayer,gaia_dr3_id,gaia_dr2_id,HD_id,HIP_id,gliese_id,2mass_id,WR_id,sp_type_hyg,sp_type_simbad
0,SIMBAD,HD 5005AB,None,,None,None,None,None,,J00524924+5637394,,None,O4V((fc))+O9.7II-III
1,SIMBAD,NGC 419 Anon 1,None,,None,None,None,None,,J01082201-7253026,,None,C3
2,SIMBAD,BD+49 435AB,None,,None,None,None,None,,J01405227+4952301,,None,K2V
3,SIMBAD,2MASS J0249-0557 c,None,,None,None,None,None,,J02495436-0558015,,None,L2
4,SIMBAD,[CBI89] 0252+0117,None,,None,None,None,None,,J02552962+0129115,,None,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,SIMBAD,Sgr A IRS 16SWE,None,,None,None,None,None,,,WR 101n,None,WC8/9
2732,SIMBAD,Sgr A IRS 21,None,,None,None,None,None,,,WR 101na,None,WC9
2733,SIMBAD,Gal Center IRS 9W,None,,None,None,None,None,,,WR 101nc,None,WN8
2734,SIMBAD,Sgr A IRS 16NE,None,,None,None,None,None,,,WR 101nd,None,Ofpe/WN9


## Query Gaia DR3 and DR2 for highest-precision astrometry


In [7]:
gaia_dr3_ids = sorted(seed["gaia_dr3_id"].dropna().astype(str).unique())
gaia_dr3_tables = []

for start in range(0, len(gaia_dr3_ids), GAIA_BATCH_SIZE):
    batch = gaia_dr3_ids[start:start + GAIA_BATCH_SIZE]
    id_list = ",".join(batch)
    query = f"""
    SELECT gs.source_id, gs.ra, gs.dec, gs.parallax, gs.parallax_error, gs.bp_rp, ap.radius_flame, ap.lum_flame
    FROM gaiadr3.gaia_source AS gs
    LEFT OUTER JOIN gaiadr3.astrophysical_parameters AS ap ON gs.source_id = ap.source_id
    WHERE gs.source_id IN ({id_list})
    """
    gaia_dr3_tables.append(Gaia.launch_job_async(query).get_results())
    print(f"Gaia DR3 rows: {min(start + GAIA_BATCH_SIZE, len(gaia_dr3_ids))}/{len(gaia_dr3_ids)}")

gaia_dr3 = vstack(gaia_dr3_tables).to_pandas() if gaia_dr3_tables else pd.DataFrame(columns=["source_id", "ra", "dec", "parallax", "parallax_error", "bp_rp", "radius_flame", "lum_flame"])
gaia_dr3["source_id"] = gaia_dr3["source_id"].astype(str)
gaia_dr3 = gaia_dr3.rename(columns={"ra": "ra_gaia_dr3", "dec": "dec_gaia_dr3", "parallax": "parallax_gaia_dr3", "parallax_error": "parallax_error_gaia_dr3", "bp_rp": "bp_rp_gaia_dr3"})

gaia_dr2_ids = sorted(seed["gaia_dr2_id"].dropna().astype(str).unique())
gaia_dr2_tables = []

for start in range(0, len(gaia_dr2_ids), GAIA_BATCH_SIZE):
    batch = gaia_dr2_ids[start:start + GAIA_BATCH_SIZE]
    id_list = ",".join(batch)
    query = f"""
    SELECT source_id, ra, dec, parallax, parallax_error, bp_rp
    FROM gaiadr2.gaia_source
    WHERE source_id IN ({id_list})
    """
    gaia_dr2_tables.append(Gaia.launch_job_async(query).get_results())
    print(f"Gaia DR2 rows: {min(start + GAIA_BATCH_SIZE, len(gaia_dr2_ids))}/{len(gaia_dr2_ids)}")

gaia_dr2 = vstack(gaia_dr2_tables).to_pandas() if gaia_dr2_tables else pd.DataFrame(columns=["source_id", "ra", "dec", "parallax", "parallax_error", "bp_rp"])
gaia_dr2["source_id"] = gaia_dr2["source_id"].astype(str)
gaia_dr2 = gaia_dr2.rename(columns={"ra": "ra_gaia_dr2", "dec": "dec_gaia_dr2", "parallax": "parallax_gaia_dr2", "parallax_error": "parallax_error_gaia_dr2", "bp_rp": "bp_rp_gaia_dr2"})

# gaia_dr3, gaia_dr2



INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR3 rows: 1000/2094
INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR3 rows: 2000/2094
INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR3 rows: 2094/2094
INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR2 rows: 1000/2041
INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR2 rows: 2000/2041
INFO: Query finished. [astroquery.utils.tap.core]
Gaia DR2 rows: 2041/2041


(                source_id  ra_gaia_dr3  dec_gaia_dr3  parallax_gaia_dr3  \
 0      211810233512673920    79.350146     45.836842          75.183786   
 1      353141694365950848    33.374663     46.339063           0.078459   
 2      400600262650253184    20.909323     49.853970           0.478638   
 3     1105375465025205120    91.639593     67.549994           0.347315   
 4     1110871270814745344   118.555032     72.227774           0.130157   
 ...                   ...          ...           ...                ...   
 2089  6861481701590665984   310.012162    -18.138756           5.171925   
 2090   692906219167392384   133.915270     27.927338           5.571939   
 2091   705156050012755712   131.674140     28.759699           9.412396   
 2092   712988829111292672   133.144229     32.474229          14.728920   
 2093  6871816939095617408   296.676011    -16.571335           0.210596   
 
       parallax_error_gaia_dr3  bp_rp_gaia_dr3  radius_flame  lum_flame  
 0          

## Query Hipparcos fallback


In [8]:
hip_ids = seed["HIP_id"].dropna().astype(int).unique()

vizier = Vizier(columns=["HIP", "RAICRS", "DEICRS", "Plx", "e_Plx", "B-V", "SpType"], row_limit=-1)
hip_all = vizier.get_catalogs("I/239/hip_main")[0].to_pandas()
hip = hip_all[hip_all["HIP"].isin(hip_ids)].copy()
hip["HIP"] = hip["HIP"].astype(str)
hip


,HIP,RAICRS,DEICRS,Plx,e_Plx,B-V,SpType
87,88,0.269160,-48.809859,5.970000,0.70,0.911,G8III
121,122,0.399379,-77.065294,14.770000,0.47,1.254,K2III
182,183,0.582976,-29.720448,6.400000,0.87,-0.150,B4V
446,447,1.341900,8.787954,1.690000,1.27,1.052,G4V:p
679,677,2.096533,29.090828,33.599998,0.73,-0.038,B9p
...,...,...,...,...,...,...,...
118129,118234,359.732181,-52.745956,12.700000,0.65,1.121,K1III
118138,118243,359.752207,55.754941,2.140000,0.75,-0.071,B1V...
118163,118268,359.827510,6.863594,30.780001,0.87,0.419,F4IV
118214,118319,359.973913,-22.428180,10.630000,1.17,0.639,G2V


## Merge sources and choose values + source columns

Priority for astrometry/parallax: Gaia DR3 → Gaia DR2 → Hipparcos → SIMBAD → HYG. Radius/luminosity from Gaia DR3 when available, else HYG luminosity. BV from Hipparcos, else HYG color index.


In [39]:
def fallback_cols(df, *cols, strip_whitespace=False):
    data = df[list(cols)].copy()

    if strip_whitespace:
        data = data.replace(r"^\s*$", np.nan, regex=True)
    else:
        data = data.replace("", np.nan)

    return data.bfill(axis=1).iloc[:, 0]

def fallback_source(df, col_source_pairs, strip_whitespace=False, default=None):
    conditions = []

    for col, _ in col_source_pairs:
        s = df[col]

        if strip_whitespace:
            cond = s.notna() & (s.astype(str).str.strip() != "")
        else:
            cond = s.notna() & (s != "")

        conditions.append(cond)

    choices = [source for _, source in col_source_pairs]

    return np.select(conditions, choices, default=default)

In [75]:
table = seed.merge(gaia_dr3, how="left", left_on="gaia_dr3_id", right_on="source_id", suffixes=("", "_gaia_dr3"))
table = table.merge(gaia_dr2, how="left", left_on="gaia_dr2_id", right_on="source_id", suffixes=("", "_gaia_dr2"))
table = table.merge(hip, how="left", left_on="HIP_id", right_on="HIP", suffixes=("", "_hip"))

table["ra"] = fallback_cols(table, "ra_gaia_dr3", "ra_gaia_dr2", "RAICRS", "ra_simbad", "ra_hyg")
table["ra_source"] = fallback_source(table, [("ra_gaia_dr3", "Gaia DR3"), ("ra_gaia_dr2", "Gaia DR2"), ("RAICRS", "Hipparcos"), ("ra_simbad", "SIMBAD"), ("ra_hyg", "HYG")])

table["dec"] = fallback_cols(table, "dec_gaia_dr3", "dec_gaia_dr2", "DEICRS", "dec_simbad", "dec_hyg")
table["dec_source"] = fallback_source(table, [("dec_gaia_dr3", "Gaia DR3"), ("dec_gaia_dr2", "Gaia DR2"), ("DEICRS", "Hipparcos"), ("dec_simbad", "SIMBAD"), ("dec_hyg", "HYG")])

table["parallax"] = fallback_cols(table, "parallax_gaia_dr3", "parallax_gaia_dr2", "Plx", "parallax_simbad", "parallax_hyg")
table["parallax_source"] = fallback_source(table, [("parallax_gaia_dr3", "Gaia DR3"), ("parallax_gaia_dr2", "Gaia DR2"), ("Plx", "Hipparcos"), ("parallax_simbad", "SIMBAD"), ("parallax_hyg", "HYG")])

table["radius"] = fallback_cols(table, "radius_flame", "radius_hyg")
table["radius_source"] = fallback_source(table, [("radius_flame", "Gaia DR3 FLAME"), ("radius_hyg", "HYG")])

table["luminosity"] = fallback_cols(table, "lum_flame", "luminosity_hyg")
table["luminosity_source"] = fallback_source(table, [("lum_flame", "Gaia DR3 FLAME"), ("luminosity_hyg", "HYG")])

table["BV_color_index"] = fallback_cols(table, "B-V", "BV_color_index_hyg")
table["BV_color_index_source"] = fallback_source(table, [("B-V", "Hipparcos"), ("BV_color_index_hyg", "HYG")])

table["sp_type"] = fallback_cols(table, "sp_type_simbad", "sp_type_hyg", "SpType", strip_whitespace=True)
table["sp_type_source"] = fallback_source(table, [("sp_type_simbad", "SIMBAD"), ("sp_type_hyg", "HYG"), ("SpType", "Hipparcos")], strip_whitespace=True)

table["dist"] = 1000.0 / table["parallax"]
table["dist_source"] = table["parallax_source"]

missing_required = table[table[["ra", "dec", "parallax", "sp_type"]].isna().any(axis=1)]
print(f"Rows missing ra/dec/parallax/sp_type before non-star filter: {len(missing_required)}")

table[["common_name", "bayer_name", "gaia_dr3_id", "gaia_dr2_id", "HD_id", "HIP_id", "gliese_id", "2mass_id", "WR_id", "ra", "ra_source", "dec", "dec_source", "parallax", "parallax_source", "sp_type", "sp_type_source", "sp_type_simbad", "sp_type_hyg", "SpType", "radius", "radius_source", "luminosity", "luminosity_source", "BV_color_index", "BV_color_index_source", "dist", "otype"]]




Rows missing ra/dec/parallax/sp_type before non-star filter: 303


,common_name,bayer_name,gaia_dr3_id,gaia_dr2_id,HD_id,HIP_id,gliese_id,2mass_id,WR_id,ra,ra_source,dec,dec_source,parallax,parallax_source,sp_type,sp_type_source,sp_type_simbad,sp_type_hyg,SpType,radius,radius_source,luminosity,luminosity_source,BV_color_index,BV_color_index_source,dist,otype
0,HD 5005AB,None,None,None,None,None,,J00524924+5637394,,13.205101,SIMBAD,56.627647,SIMBAD,NaN,None,O4V((fc))+O9.7II-III,SIMBAD,O4V((fc))+O9.7II-III,None,NaN,NaN,None,NaN,None,NaN,None,NaN,**
1,NGC 419 Anon 1,None,None,None,None,None,,J01082201-7253026,,17.091750,SIMBAD,-72.884083,SIMBAD,NaN,None,C3,SIMBAD,C3,None,NaN,NaN,None,NaN,None,NaN,None,NaN,**
2,BD+49 435AB,None,None,None,None,None,,J01405227+4952301,,25.217825,SIMBAD,49.874986,SIMBAD,NaN,None,K2V,SIMBAD,K2V,None,NaN,NaN,None,NaN,None,NaN,None,NaN,**
3,2MASS J0249-0557 c,None,None,None,None,None,,J02495436-0558015,,42.476526,SIMBAD,-5.967106,SIMBAD,20.1,SIMBAD,L2,SIMBAD,L2,None,NaN,NaN,None,NaN,None,NaN,None,49.751244,BD*
4,[CBI89] 0252+0117,None,None,None,None,None,,J02552962+0129115,,43.873456,SIMBAD,1.486531,SIMBAD,NaN,None,D,SIMBAD,D,None,NaN,NaN,None,NaN,None,NaN,None,NaN,**
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,Sgr A IRS 16SWE,None,None,None,None,None,,,WR 101n,266.417420,SIMBAD,-29.008127,SIMBAD,NaN,None,WC8/9,SIMBAD,WC8/9,None,NaN,NaN,None,NaN,None,NaN,None,NaN,WR*
2732,Sgr A IRS 21,None,None,None,None,None,,,WR 101na,266.417590,SIMBAD,-29.008569,SIMBAD,NaN,None,WC9,SIMBAD,WC9,None,NaN,NaN,None,NaN,None,NaN,None,NaN,WR*
2733,Gal Center IRS 9W,None,None,None,None,None,,,WR 101nc,266.417750,SIMBAD,-29.009361,SIMBAD,NaN,None,WN8,SIMBAD,WN8,None,NaN,NaN,None,NaN,None,NaN,None,NaN,WR*
2734,Sgr A IRS 16NE,None,None,None,None,None,,,WR 101nd,266.417750,SIMBAD,-29.007556,SIMBAD,NaN,None,Ofpe/WN9,SIMBAD,Ofpe/WN9,None,NaN,NaN,None,NaN,None,NaN,None,NaN,WR*


In [76]:
# Split non-stars using SIMBAD otype.
# Keep stellar SIMBAD object types; move clearly non-stellar object types out.
otype_text = table["otype"].fillna("").astype(str).str.strip()

stellar_otype = otype_text.str.contains(
    r"\*|WD|WR|RGB|AGB|C\*|Y\*O|TT\*|Be\*|Em\*|pA\*|pr\*|PMS\*|post-AGB|HB\*|RC\*|BY\*|RS\*|SN\*|LM\*|brownD\*",
    regex=True,
    case=False,
)

nonstellar_otype = otype_text.str.contains(
    r"^(?:G|Galaxy|AGN|QSO|BLLac|Seyfert|SyG|ClG|Gin|GroupG|Cl|GlCl|PN|HII|SNR|ISM|Radio|X|gamma|Mas|MolCld|Dense|Cloud|Neb|RfN)",
    regex=True,
    case=False,
)

# Blank otype stays in main table for now; explicit non-stellar otype moves out unless SIMBAD also marks it stellar.
non_star_mask = nonstellar_otype & ~stellar_otype

non_stars = table[non_star_mask].copy()
table = table[~non_star_mask].copy()

# remove stars with missing location data
missing_required_mask = table[["ra", "dec", "parallax", "sp_type"]].isna().any(axis=1)
missing_required = table[missing_required_mask]
table = table[~missing_required_mask].copy()

print(f"Non-stars moved out by SIMBAD otype: {len(non_stars)}")
print(f"Star rows remaining: {len(table)}")
print(f"Star rows missing ra/dec/parallax/sp_type: {len(missing_required)}")

non_stars[["common_name", "bayer_name", "HD_id", "HIP_id", "otype", "sp_type", "ra", "dec", "parallax"]]



Non-stars moved out by SIMBAD otype: 36
Star rows remaining: 2406
Star rows missing ra/dec/parallax/sp_type: 294


,common_name,bayer_name,HD_id,HIP_id,otype,sp_type,ra,dec,parallax
39,WKK136-337,None,None,None,PN,O(H)3III-V,237.620458,-59.980556,NaN
60,Gomez's Hamburger,None,None,None,PN?,A0III:,272.305824,-32.180565,NaN
68,PN Kn 15,None,None,None,PN?,PG1159,295.168833,29.502417,NaN
115,PN SkAc 1,None,None,None,PN,O8/O9,214.091409,13.873352,1.504255
155,Jacoby 1,None,None,None,PN,PG1159,230.443987,52.367791,1.295383
165,Sunflower Nebula,None,164963,None,PN,[WC],269.639178,66.632957,0.732126
179,Blue Flash Nebula,None,193949,None,PN,[WC3],305.595767,20.104499,0.368093
181,PN Kn 12,None,None,None,PN?,PG1159,300.843907,21.597858,0.334072
183,Dumbbell Nebula,None,None,None,PN,DAO.6,299.901564,22.721214,2.570049
184,Kronberger PN J1944.9+2245,None,None,None,PN,DA1,296.246358,22.763181,2.549663


### Requires manual editing

In [47]:
QTable.from_pandas(missing_required).show_in_browser(jsviewer=True)

In [13]:
QTable.from_pandas(table[["sp_type_simbad", "sp_type_hyg", "SpType"]]).show_in_browser(jsviewer=True)

## Classify spectra


In [77]:
# Pick most informative spectral type, then classify each component.
# Multiple stars: split on + and join output classifications with " + ".

sp_source_cols = [("sp_type_simbad", "SIMBAD"), ("sp_type_hyg", "HYG"), ("SpType", "Hipparcos")]

best_sp_types = []
best_sp_sources = []

for _, row in table.iterrows():
    candidates = []
    for col, source in sp_source_cols:
        value = row.get(col)
        if pd.isna(value) or str(value).strip() == "":
            continue
        text = str(value).strip()
        clean = re.sub(r"\s+", "", text)
        score = len(clean)
        score += 20 if re.search(r"(Ia\+|Ia|Iab|Ib|III|II|IV|V|VI|VII|sd|esd|usd|D[A-Z]|W[NCOR]|C[\-A-Z]?|CH|CN)", clean, re.I) else 0
        score += 10 if "+" in clean else 0
        score += 5 if re.search(r"[epnm:]|var|shell", clean, re.I) else 0
        candidates.append((score, source, text))

    if candidates:
        score, source, text = sorted(candidates, key=lambda item: item[0], reverse=True)[0]
        best_sp_types.append(text)
        best_sp_sources.append(source)
    else:
        best_sp_types.append(None)
        best_sp_sources.append(None)

table["sp_type"] = best_sp_types
table["sp_type_source"] = best_sp_sources

COLOR_BY_CLASS = {
    "O": "blue", "B": "blue", "A": "white", "F": "yellow-white", "G": "yellow", "K": "orange", "M": "red",
    "L": "brown", "T": "brown", "Y": "brown",
}
WD_SUBTYPES = {
    "DA": "hydrogen-atmosphere white dwarf",
    "DB": "helium-atmosphere white dwarf",
    "DC": "featureless white dwarf",
    "DO": "ionized-helium white dwarf",
    "DQ": "carbon-feature white dwarf",
    "DZ": "metal-line white dwarf",
    "DX": "unknown-spectrum white dwarf",
}
WR_SUBTYPES = {
    "WN": "nitrogen-sequence Wolf-Rayet star",
    "WC": "carbon-sequence Wolf-Rayet star",
    "WO": "oxygen-sequence Wolf-Rayet star",
}

spectral_classes = []
mk_classes = []
star_types = []
spectral_peculiarities = []

for sp_value, wr_id, otype in zip(table["sp_type"], table["WR_id"], table["otype"]):
    if pd.isna(sp_value) or str(sp_value).strip() == "":
        spectral_classes.append(None)
        mk_classes.append("unknown")
        star_types.append("unknown")
        spectral_peculiarities.append(None)
        continue

    raw = str(sp_value).strip()
    otype_clean = "" if pd.isna(otype) else str(otype).strip()
    wr_id_clean = "" if pd.isna(wr_id) else str(wr_id).strip()
    otype_is_wr = bool(re.search(r"WR", otype_clean, re.I))
    otype_is_wd = bool(re.search(r"WD", otype_clean, re.I))
    otype_is_carbon = bool(re.search(r"C\*", otype_clean, re.I))
    components = [part for part in re.split(r"\s*\+\s*", raw) if part]
    comp_classes = []
    comp_mk = []
    comp_stage = []
    comp_pec = []

    for component in components:
        # Slash means between possible classes; use first/best option.
        # Preserve luminosity suffix for compact forms like M2/3V -> M2V.
        comp_text = component.strip()
        comp_text = re.sub(r"^\[([^\]]+)\]", r"\1", comp_text)
        slash_subtype = re.match(r"^((?:sd|esd|usd|d)?[OBAFGKMLTY][0-9.]*)/[OBAFGKMLTY]?[0-9.]+(.*)$", comp_text, re.I)
        if slash_subtype:
            comp = slash_subtype.group(1) + slash_subtype.group(2)
        else:
            comp = re.split(r"/", comp_text)[0]
        comp = re.sub(r"\s+", "", comp)
        comp_upper = comp.upper()
        pec = []

        if ":" in comp:
            pec.append("uncertain")
        if re.search(r"VAR", comp, re.I):
            pec.append("variable spectrum")
        if re.search(r"SHELL", comp, re.I):
            pec.append("shell star")

        wr_match = re.match(r"^(W[NCOR])([0-9.]*)?([A-Z]*)", comp_upper)
        if wr_match or otype_is_wr or wr_id_clean != "":
            wr_code = wr_match.group(1) if wr_match else "WR"
            wr_family = wr_code[:2] if wr_code.startswith("W") else "WR"
            comp_classes.append("WR")
            comp_mk.append("wolf-rayet")
            comp_stage.append(WR_SUBTYPES.get(wr_family, "Wolf-Rayet star"))
            if wr_match and wr_match.group(3):
                suffix = wr_match.group(3).lower()
                if "H" in wr_match.group(3): pec.append("hydrogen present")
                if "A" in wr_match.group(3): pec.append("absorption lines")
                if "B" in wr_match.group(3): pec.append("broad emission lines")
            comp_pec.append(", ".join(dict.fromkeys(pec)) if pec else None)
            continue

        wd_match = re.match(r"^(D[A-Z]?)([0-9.]*)", comp_upper)
        if wd_match or otype_is_wd:
            wd_code = wd_match.group(1) if wd_match else "D"
            comp_classes.append("D")
            comp_mk.append("white dwarf")
            comp_stage.append(WD_SUBTYPES.get(wd_code, "white dwarf"))
            if "P" in comp_upper: pec.append("magnetic/polarized")
            if "H" in comp_upper: pec.append("magnetic")
            if "V" in comp_upper: pec.append("variable")
            comp_pec.append(", ".join(dict.fromkeys(pec)) if pec else None)
            continue

        carbon_match = re.match(r"^(C(?:-?[A-Z])?|CH|CN|R|N)", comp_upper)
        if carbon_match or otype_is_carbon:
            carbon_code = carbon_match.group(1).replace("-", "") if carbon_match else "C"
            comp_classes.append("C")
            comp_mk.append("carbon star")
            if carbon_code == "CH":
                comp_stage.append("CH carbon star")
            elif carbon_code in ["CN", "N"]:
                comp_stage.append("C-N carbon star")
            elif carbon_code == "R":
                comp_stage.append("C-R carbon star")
            else:
                comp_stage.append("carbon star")
            comp_pec.append(", ".join(dict.fromkeys(pec)) if pec else None)
            continue

        subdwarf_match = re.match(r"^(ESD|USD|SD)", comp_upper)
        is_subdwarf = bool(subdwarf_match)
        is_dwarf_prefix = bool(re.match(r"^D[OBAFGKMLTY]", comp_upper))
        class_match = re.match(r"^(?:ESD|USD|SD|D)?([OBAFGKMLTY])", comp_upper)
        spec_class = class_match.group(1) if class_match else None
        color = COLOR_BY_CLASS.get(spec_class, None)

        if spec_class in ["L", "T", "Y"] and not re.search(r"IV|V|III|II|I", comp_upper):
            mk = "brown dwarf"
            stage = f"{spec_class} dwarf"
        elif is_subdwarf or re.search(r"VI", comp_upper):
            if subdwarf_match and subdwarf_match.group(1) == "ESD":
                mk = "extreme subdwarf"
                stage = f"extreme {color} subdwarf" if color else "extreme subdwarf"
            elif subdwarf_match and subdwarf_match.group(1) == "USD":
                mk = "ultra subdwarf"
                stage = f"ultra {color} subdwarf" if color else "ultra subdwarf"
            else:
                mk = "subdwarf"
                stage = f"{color} subdwarf" if color else "subdwarf"
        elif re.search(r"IA\+|0", comp_upper):
            mk = "hypergiant"
            stage = f"{color} hypergiant" if color else "hypergiant"
        elif re.search(r"(?<!I)I(?:A|AB|B)?(?!I|V)", comp_upper):
            mk = "supergiant"
            stage = f"{color} supergiant" if color else "supergiant"
        elif re.search(r"II", comp_upper):
            mk = "bright giant"
            stage = f"{color} bright giant" if color else "bright giant"
        elif re.search(r"III", comp_upper):
            mk = "giant"
            stage = f"{color} giant" if color else "giant"
        elif re.search(r"IV", comp_upper):
            mk = "subgiant"
            stage = f"{color} subgiant" if color else "subgiant"
        elif is_dwarf_prefix or re.search(r"(?<!I)V(?!I)", comp_upper):
            mk = "main-sequence"
            if spec_class == "M":
                stage = "red dwarf"
            elif color:
                stage = f"{color} main-sequence star"
            else:
                stage = "main-sequence star"
        else:
            mk = "unknown"
            stage = f"{color} star" if color else "unknown"

        if re.search(r"E", comp): pec.append("emission lines")
        if re.search(r"P", comp): pec.append("peculiar")
        if re.search(r"N", comp): pec.append("broad/nebulous lines")
        if re.search(r"M", comp) and spec_class != "M": pec.append("metallic-lined")
        if re.search(r"K", comp) and spec_class not in ["K"]: pec.append("interstellar/calcium K anomaly")

        comp_classes.append(spec_class)
        comp_mk.append(mk)
        comp_stage.append(stage)
        comp_pec.append(", ".join(dict.fromkeys(pec)) if pec else None)

    spectral_classes.append(" + ".join([value for value in comp_classes if value]) or None)
    mk_classes.append(" + ".join([value for value in comp_mk if value]) or "unknown")
    star_types.append(" + ".join([value for value in comp_stage if value]) or "unknown")
    spectral_peculiarities.append(" + ".join([value for value in comp_pec if value]) or None)

table["spectral_class"] = spectral_classes
table["luminosity_class"] = mk_classes
table["star_type"] = star_types
table["spectral_peculiarities"] = spectral_peculiarities

# # Remove anything still classified as unknown after spectral classification.
# # Keep it separately for later review.
# unknown_classification_mask = table["luminosity_class"].fillna("").str.contains(r"\bunknown\b", regex=True)
# unknown_classification = table[unknown_classification_mask].copy()
# table = table[~unknown_classification_mask].copy()

# print(f"Rows moved out due unknown spectral classification: {len(unknown_classification)}")
print(f"Rows remaining after classification filter: {len(table)}")

# Display names after classification: common name -> ASCII Bayer -> lookup name -> type-specific catalog convention -> generic IDs.
GREEK_ASCII = {
    "α": "Alpha", "β": "Beta", "γ": "Gamma", "δ": "Delta", "ε": "Epsilon", "ζ": "Zeta",
    "η": "Eta", "θ": "Theta", "ι": "Iota", "κ": "Kappa", "λ": "Lambda", "μ": "Mu",
    "ν": "Nu", "ξ": "Xi", "ο": "Omicron", "π": "Pi", "ρ": "Rho", "σ": "Sigma",
    "τ": "Tau", "υ": "Upsilon", "φ": "Phi", "χ": "Chi", "ψ": "Psi", "ω": "Omega",
}
SUPERSCRIPT_ASCII = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
GREEK_ABBR_ASCII = {
    "alf": "Alpha", "bet": "Beta", "gam": "Gamma", "del": "Delta", "eps": "Epsilon", "zet": "Zeta",
    "eta": "Eta", "the": "Theta", "iot": "Iota", "kap": "Kappa", "lam": "Lambda", "mu.": "Mu",
    "nu.": "Nu", "ksi": "Xi", "omi": "Omicron", "pi.": "Pi", "rho": "Rho", "sig": "Sigma",
    "tau": "Tau", "ups": "Upsilon", "phi": "Phi", "khi": "Chi", "psi": "Psi", "ome": "Omega",
}

bayer_name_ascii = []
for bayer_name, simbad_bayer in zip(table["bayer_name"], table["simbad_bayer"]):
    ascii_name = bayer_name
    if isinstance(bayer_name, str):
        for greek, english in GREEK_ASCII.items():
            ascii_name = ascii_name.replace(greek, english)
        ascii_name = ascii_name.translate(SUPERSCRIPT_ASCII)

    if isinstance(simbad_bayer, str):
        match = re.match(r"^([a-z.]{3})(\d{0,2})\s+([A-Z][A-Za-z]{2})(?:\s+([A-Z]))?$", simbad_bayer)
        if match:
            greek_abbr, number, constellation_abbr, component = match.groups()
            english = GREEK_ABBR_ASCII.get(greek_abbr)
            constellation = CONSTELLATION_GENITIVE.get(constellation_abbr)
            if english and constellation and component:
                ascii_name = f"{english} {constellation} {component}"
    bayer_name_ascii.append(ascii_name)

table["bayer_name_ascii"] = bayer_name_ascii

weird_common_name = table["common_name"].fillna("").str.match(
    r"^(Gaia|HD|HIP|2MASS|WISE|TYC|TIC|UCAC|USNO|GJ|Gl|LP|LHS|LTT|BD|CD|CPD|SAO|IRAS|ASAS|SDSS|2MUCD|\[)",
    case=False,
)

lookup_name = table["lookup_name"] if "lookup_name" in table.columns else pd.Series(index=table.index, dtype=object)
type_specific_id = pd.Series(index=table.index, dtype=object)
wr_id_nonempty = table["WR_id"].notna() & (table["WR_id"].astype(str).str.strip() != "")
type_specific_id = type_specific_id.mask(table["luminosity_class"].str.contains("wolf-rayet", na=False) & wr_id_nonempty, table["WR_id"])
if "main_id" in table.columns:
    type_specific_id = type_specific_id.mask(table["luminosity_class"].str.contains("white dwarf", na=False) & table["main_id"].notna(), table["main_id"])
type_specific_id = type_specific_id.mask(table["spectral_class"].fillna("").str.contains(r"\b[LTY]\b", regex=True) & table["2mass_id"].notna(), "2MASS " + table["2mass_id"].astype(str))

fallback_id = pd.Series(index=table.index, dtype=object)
fallback_id = fallback_id.combine_first(type_specific_id)
fallback_id = fallback_id.combine_first("Gaia DR3 " + table["gaia_dr3_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first("Gaia DR2 " + table["gaia_dr2_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first("HD " + table["HD_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first("HIP " + table["HIP_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first(table["gliese_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first("2MASS " + table["2mass_id"].dropna().astype(str))
fallback_id = fallback_id.combine_first(table["WR_id"].dropna().astype(str))

table["display_name"] = table["common_name"]
table.loc[table["display_name"].isna() | (table["display_name"].str.strip() == ''), "display_name"] = table["bayer_name_ascii"]
table["display_name"] = table["display_name"].combine_first(lookup_name).combine_first(fallback_id)

table[["display_name", "common_name", "bayer_name", "bayer_name_ascii", "sp_type", "sp_type_source", "spectral_class", "luminosity_class", "star_type", "spectral_peculiarities"]]


Rows remaining after classification filter: 2406


,display_name,common_name,bayer_name,bayer_name_ascii,sp_type,sp_type_source,spectral_class,luminosity_class,star_type,spectral_peculiarities
3,2MASS J0249-0557 c,2MASS J0249-0557 c,None,None,L2,SIMBAD,L,brown dwarf,L dwarf,None
32,VHS J1256-1257B,VHS J1256-1257B,None,None,L7,SIMBAD,L,brown dwarf,L dwarf,None
35,LHS 2803 B,LHS 2803 B,None,None,T5.5,SIMBAD,T,brown dwarf,T dwarf,None
74,Haro 6-10 IRC,Haro 6-10 IRC,None,None,K7,SIMBAD,K,unknown,orange star,None
75,BD+26 4251BC,BD+26 4251BC,None,None,esdM1,SIMBAD,M,extreme subdwarf,extreme red subdwarf,None
...,...,...,...,...,...,...,...,...,...,...
2676,CFBDSIR 1458+1013B,CFBDSIR 1458+1013B,None,None,T8+/Y?,SIMBAD,T,brown dwarf + unknown,T dwarf + unknown,None
2677,CFBDSIR 1458+1013A,CFBDSIR 1458+1013A,None,None,T8+/Y?,SIMBAD,T,brown dwarf + unknown,T dwarf + unknown,None
2680,LHS 6176B,LHS 6176B,None,None,T8p,SIMBAD,T,brown dwarf,T dwarf,None
2696,COCONUTS-2b,COCONUTS-2b,None,None,T9,SIMBAD,T,brown dwarf,T dwarf,None


## Prepare JSON export

Writes schema to `data/star.schema.json`. Export cell below is prepared but commented; uncomment write loop when ready.


In [82]:
STAR_OUTPUT_DIR = Path.cwd().parent / "data" / "stars" if Path.cwd().name == "scripts" else Path.cwd() / "data" / "stars"

export_columns = [
    "display_name", "common_name", "bayer_name", "bayer_name_ascii", "gaia_dr3_id", "gaia_dr2_id", "HD_id", "HIP_id",
    "gliese_id", "2mass_id", "WR_id", "ra", "ra_source", "dec", "dec_source", "parallax", "parallax_source",
    "dist", "sp_type", "sp_type_source", "spectral_class", "luminosity_class", "star_type", "spectral_peculiarities",
    "radius", "radius_source", "luminosity", "luminosity_source", "BV_color_index", "BV_color_index_source"
]

stars_for_export = table[export_columns].copy()
stars_for_export = stars_for_export.replace({np.nan: None})

STAR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

written_paths = []
for idx, row in stars_for_export.iterrows():
    obj = {key: row[key] for key in export_columns}

    base_id = (
        obj["display_name"]
        or obj.get("gaia_dr3_id")
        or obj.get("gaia_dr2_id")
        or obj.get("HIP_id")
        or obj.get("HD_id")
        or obj.get("gliese_id")
        or obj.get("2mass_id")
        or obj.get("WR_id")
    )
    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(base_id)).strip("_").lower()
    path = STAR_OUTPUT_DIR / f"{safe_id}.json"

    suffix = 2
    while path.exists():
        try:
            existing = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            existing = None
        if existing == obj:
            break
        path = STAR_OUTPUT_DIR / f"{safe_id}_{suffix}.json"
        suffix += 1

    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    written_paths.append(path)

print(f"Wrote {len(written_paths)} JSON files to {STAR_OUTPUT_DIR}")
written_paths[:10]



Wrote 2406 JSON files to /home/sebl/code/etoile-data/data/stars


[PosixPath('/home/sebl/code/etoile-data/data/stars/2mass_j0249-0557_c.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/vhs_j1256-1257b.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/lhs_2803_b.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/haro_6-10_irc.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/bd_26_4251bc.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/bd_31_3330bc.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/hd_55271ab.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/iota_leonis_b.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/k2-288b.json'),
 PosixPath('/home/sebl/code/etoile-data/data/stars/mu2_herculis.json')]

In [73]:
# Rows where spectral types from HYG, SIMBAD, and Hipparcos disagree.
# Compares normalized non-empty values; row is mismatch when at least 2 sources exist and not all equal.

spectral_compare = table.copy()

spectral_compare["sp_type_simbad_norm"] = spectral_compare["sp_type_simbad"].fillna("").astype(str).str.replace(" ", "", regex=False)
spectral_compare["sp_type_hyg_norm"] = spectral_compare["sp_type_hyg"].fillna("").astype(str).str.replace(" ", "", regex=False)
spectral_compare["sp_type_hip_norm"] = spectral_compare["SpType"].fillna("").astype(str).str.replace(" ", "", regex=False)

spectral_values = spectral_compare[["sp_type_simbad_norm", "sp_type_hyg_norm", "sp_type_hip_norm"]]
spectral_source_count = spectral_values.ne("").sum(axis=1)
spectral_unique_count = spectral_values.replace("", np.nan).nunique(axis=1)

spectral_mismatch = spectral_compare[
    (spectral_source_count >= 2)
    & (spectral_unique_count > 1)
].copy()

spectral_mismatch = spectral_mismatch[[
    "display_name",
    "common_name",
    "bayer_name",
    "bayer_name_ascii",
    "sp_type",
    "sp_type_source",
    "sp_type_simbad",
    "sp_type_hyg",
    "SpType",
]]

print(f"Spectral type mismatches: {len(spectral_mismatch)}")
QTable.from_pandas(spectral_mismatch).show_in_browser(jsviewer=True)



Spectral type mismatches: 1076
